In [8]:
import mpmath
import numpy as np
import sympy as sp
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import gaussian_kde
from scipy.integrate import odeint
sp.init_printing(use_latex=True)

In [9]:
def generar_base_24(limite):
    """
    Toma una recta numérica del 1 al 'limite' y la proyecta en el espacio Tetravigesimal.
    Retorna un diccionario con los números, su estado primo, y su posición en la Base 24.
    """
    numeros = np.arange(1, limite + 1)
    es_primo = np.array([sp.isprime(int(n)) for n in numeros])
    posicion_base_24 = numeros % 24
    return {
        "n": numeros,
        "es_primo": es_primo,
        "base_24": posicion_base_24
    }
datos = generar_base_24(100)
print(f"Número 97: ¿Es primo? {datos['es_primo'][96]} | Posición en Base 24: {datos['base_24'][96]}")

Número 97: ¿Es primo? True | Posición en Base 24: 1


In [10]:
t = np.linspace(0, 20, 1000)
x = t * np.cos(t)
y = t * np.sin(t)
z = t
fig = go.Figure(data=[go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color='orange', width=4))])
fig.update_layout(
    title='Atractor de Prueba (Topología Base 24)',
    scene=dict(xaxis_title='Eje X', yaxis_title='Eje Y', zaxis_title='Eje Z'),
    template='plotly_dark'
)
fig.show()

In [11]:
limite = 500
paso = 5
n_vals = np.arange(1, limite + 1)
primos = np.array([sp.isprime(int(i)) for i in n_vals])
angulos = (n_vals % 24) * 15
radios = (n_vals / 24) + 5
fig = go.Figure()
fig.add_trace(go.Scatterpolar(
    r=[], theta=[], mode='markers',
    marker=dict(color='rgba(255, 255, 255, 0.15)', size=4),
    name='No Primo', hoverinfo='text'
))
fig.add_trace(go.Scatterpolar(
    r=[], theta=[], mode='markers',
    marker=dict(color='#FF8C00', size=8, symbol='circle', line=dict(color='white', width=0.5)),
    name='Primo (Atractor)', hoverinfo='text'
))
frames = []
for k in range(paso, limite + paso, paso):
    k = min(k, limite)
    idx = np.arange(0, k)
    idx_no_primos = idx[~primos[idx]]
    idx_primos = idx[primos[idx]]
    frame = go.Frame(
        data=[
            go.Scatterpolar(
                r=radios[idx_no_primos], theta=angulos[idx_no_primos],
                text=[f"N: {n_vals[i]}" for i in idx_no_primos]
            ),
            go.Scatterpolar(
                r=radios[idx_primos], theta=angulos[idx_primos],
                text=[f"N: {n_vals[i]} [PRIMO]" for i in idx_primos]
            )
        ],
        name=str(k)
    )
    frames.append(frame)
fig.frames = frames
sliders = [{
    "pad": {"b": 10, "t": 50},
    "len": 0.9, "x": 0.1, "y": 0,
    "currentvalue": {"font": {"size": 18, "color": "#FF8C00"}, "prefix": "Límite actual N = ", "visible": True, "xanchor": "right"},
    "steps": [{"args": [[f.name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}], "label": f.name, "method": "animate"} for f in frames]
}]
fig.update_layout(
    title='Formación de los 8 Atractores (Base Guzmánica)',
    template='plotly_dark',
    width=800,
    height=800,
    polar=dict(
        angularaxis=dict(tickmode='array', tickvals=np.arange(0, 360, 15), ticktext=np.arange(0, 24)),
        radialaxis=dict(visible=False, range=[0, (limite/24) + 6])
    ),
    updatemenus=[{
        "buttons": [
            {"args": [None, {"frame": {"duration": 150, "redraw": True}, "fromcurrent": True}], "label": "▶ Reproducir", "method": "animate"},
            {"args": [[None], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}], "label": "⏸ Pausa", "method": "animate"}
        ],
        "direction": "left", "pad": {"r": 10, "t": 87}, "showactive": False, "type": "buttons", "x": 0.1, "xanchor": "right", "y": 0, "yanchor": "top"
    }],
    sliders=sliders
)
fig.show()

In [12]:
mpmath.mp.dps = 15
def renderizar_zeta_guzmanica(resolucion_x=60, resolucion_y=200):
    print("Calculando matriz compleja... Esto puede tomar unos segundos.")
    x_vals = np.linspace(0, 1.2, resolucion_x)
    y_vals = np.linspace(10, 50, resolucion_y)
    X, Y = np.meshgrid(x_vals, y_vals)
    Z_superficie = np.zeros_like(X)
    Z_colores_base24 = np.zeros_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            x = X[i, j]
            y = Y[i, j]
            s = complex(x, y)
            zeta_val = mpmath.zeta(s)
            magnitud_zeta = float(abs(zeta_val))
            fase_zeta = float(mpmath.arg(zeta_val))
            eml = np.exp(x) - np.log(y)
            Z_superficie[i, j] = magnitud_zeta * abs(eml)
            carril = ((fase_zeta + np.pi) / (2 * np.pi)) * 24
            Z_colores_base24[i, j] = np.floor(carril)
    fig = go.Figure(data=[go.Surface(
        x=X, y=Y, z=Z_superficie,
        surfacecolor=Z_colores_base24,
        colorscale='Turbo',
        cmin=0, cmax=23,
        colorbar=dict(title='Atractor (0-23)', tickmode='array', tickvals=np.arange(0, 24, 2))
    )])
    fig.update_layout(
        title='Función Zeta de Riemann + Operador EML bajo Base 24',
        scene=dict(
            xaxis_title='Re(s) [x]',
            yaxis_title='Im(s) [y]',
            zaxis_title='|Zeta| deformado por EML',
            camera=dict(eye=dict(x=-1.5, y=-1.5, z=1.5)) # Ángulo de cámara táctico
        ),
        template='plotly_dark',
        width=900, height=800
    )
    print("Topología generada con éxito.")
    fig.show()
renderizar_zeta_guzmanica()

Calculando matriz compleja... Esto puede tomar unos segundos.
Topología generada con éxito.


In [13]:
mpmath.mp.dps = 15
def osciloscopio_guzmanico():
    print("Calibrando osciloscopio en Re(s) = 0.5... Escaneando frecuencias.")
    x_critico = 0.5
    y_vals = np.linspace(10, 50, 3000)
    atractores = [1, 5, 7, 11, 13, 17, 19, 23]
    frecuencias_y = []
    nivel_atractor = []
    energia_eml = []
    for y in y_vals:
        s = complex(x_critico, y)
        zeta_val = mpmath.zeta(s)
        magnitud = float(abs(zeta_val))
        fase = float(mpmath.arg(zeta_val))
        carril = int(np.floor(((fase + np.pi) / (2 * np.pi)) * 24))
        if carril in atractores:
            frecuencias_y.append(y)
            nivel_atractor.append(str(carril))
            eml = abs(np.exp(x_critico) - np.log(y))
            energia_eml.append(magnitud * eml)
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=frecuencias_y,
        y=nivel_atractor,
        mode='markers',
        marker=dict(
            size=8,
            color=energia_eml,
            colorscale='Inferno',
            showscale=True,
            colorbar=dict(title='Energía (EML x |Zeta|)'),
            line=dict(width=0)
        ),
        text=[f"Frecuencia (y): {y:.2f}<br>Energía: {e:.4f}" for y, e in zip(frecuencias_y, energia_eml)],
        hoverinfo='text'
    ))
    fig.update_layout(
        title='Osciloscopio Cuántico: Saltos entre los 8 Atractores Base 24',
        xaxis_title='Frecuencia / Tiempo (Eje Imaginario "y")',
        yaxis_title='Nivel de Energía Cuantizado (Atractor)',
        yaxis=dict(type='category', categoryorder='array', categoryarray=[str(a) for a in atractores]),
        template='plotly_dark',
        height=600
    )
    print("Señal interceptada con éxito.")
    fig.show()
osciloscopio_guzmanico()

Calibrando osciloscopio en Re(s) = 0.5... Escaneando frecuencias.
Señal interceptada con éxito.


In [14]:
print("Extrayendo los primeros 100 ceros no triviales de Riemann...")
mpmath.mp.dps = 15
ceros_y = [float(mpmath.zetazero(i).imag) for i in range(1, 101)]
espaciamientos = np.diff(ceros_y)
espaciamientos_norm = espaciamientos / np.mean(espaciamientos)
s_vals = np.linspace(0, 3, 500)
wigner_gue = (32 / np.pi**2) * (s_vals**2) * np.exp(-(4 / np.pi) * (s_vals**2))
kde = gaussian_kde(espaciamientos_norm)
riemann_densidad = kde(s_vals)
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=s_vals, y=wigner_gue,
    mode='lines', name='Matemática Cuántica (GUE)',
    line=dict(color='#00ffcc', width=3, dash='dash')
))
fig.add_trace(go.Scatter(
    x=s_vals, y=riemann_densidad,
    mode='lines', name='Ceros de Riemann (Empírico)',
    line=dict(color='#ff00ff', width=3),
    fill='tozeroy', fillcolor='rgba(255, 0, 255, 0.2)'
))
fig.add_trace(go.Histogram(
    x=espaciamientos_norm, histnorm='probability density',
    name='Distancia entre Ceros',
    marker_color='rgba(255, 140, 0, 0.6)',
    xbins=dict(start=0, end=3, size=0.2)
))
fig.update_layout(
    title='Correlación Montgomery-Dyson: Riemann vs Mecánica Cuántica',
    xaxis_title='Distancia normalizada entre Ceros (s)',
    yaxis_title='Probabilidad P(s)',
    template='plotly_dark',
    barmode='overlay',
    legend=dict(x=0.6, y=0.9)
)
print("Análisis de matrices completado.")
fig.show()

Extrayendo los primeros 100 ceros no triviales de Riemann...
Análisis de matrices completado.


In [15]:
mpmath.mp.dps = 15
def ejecutar_maquina_guzmanica(N_ceros=1000):
    print(f"Iniciando Máquina Guzmánica para los primeros {N_ceros} ciclos...")
    y_vals = [float(mpmath.zetazero(i).imag) for i in range(1, N_ceros + 1)]
    x_critico = 0.5
    atractores_validos = [1, 5, 7, 11, 13, 17, 19, 23]
    conteo_atractores = {a: 0 for a in atractores_validos}
    fases_registradas = []
    energias_registradas = []
    for y in y_vals:
        s = complex(x_critico, y)
        zeta_val = mpmath.zeta(s)
        fase = float(mpmath.arg(zeta_val))
        carril = int(np.floor(((fase + np.pi) / (2 * np.pi)) * 24))
        energia_eml = abs(np.exp(x_critico) - np.log(y))
        fases_registradas.append(carril)
        energias_registradas.append(energia_eml)
        if carril in atractores_validos:
            conteo_atractores[carril] += 1
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=[str(a) for a in atractores_validos],
        y=[conteo_atractores[a] for a in atractores_validos],
        marker=dict(
            color=[conteo_atractores[a] for a in atractores_validos],
            colorscale='Turbo',
            line=dict(color='white', width=1)
        ),
        name='Impactos por Atractor'
    ))
    fig.update_layout(
        title=f'Macro-Estado: Distribución de {N_ceros} Ceros en la Topología Base 24',
        xaxis_title='Carriles Cuantizados (Atractores)',
        yaxis_title='Cantidad de Impactos (Ceros anclados)',
        template='plotly_dark',
        height=500
    )
    print("Simulación a gran escala completada.")
    fig.show()
ejecutar_maquina_guzmanica(1000)

Iniciando Máquina Guzmánica para los primeros 1000 ciclos...
Simulación a gran escala completada.


In [16]:
mpmath.mp.dps = 15
def matriz_transicion_guzmanica(N_ceros=1000):
    print(f"Calculando cadena de Markov para {N_ceros} saltos cuánticos...")
    y_vals = [float(mpmath.zetazero(i).imag) for i in range(1, N_ceros + 1)]
    atractores_validos = [1, 5, 7, 11, 13, 17, 19, 23]
    indice_atractor = {a: i for i, a in enumerate(atractores_validos)}
    matriz_saltos = np.zeros((8, 8))
    estado_previo = None
    for y in y_vals:
        s = complex(0.5, y)
        fase = float(mpmath.arg(mpmath.zeta(s)))
        carril = int(np.floor(((fase + np.pi) / (2 * np.pi)) * 24))
        if carril in atractores_validos:
            if estado_previo is not None:
                i = indice_atractor[estado_previo]
                j = indice_atractor[carril]
                matriz_saltos[i, j] += 1
            estado_previo = carril
    suma_filas = matriz_saltos.sum(axis=1, keepdims=True)
    matriz_probabilidades = np.divide(matriz_saltos, suma_filas, out=np.zeros_like(matriz_saltos), where=suma_filas!=0) * 100
    etiquetas = [str(a) for a in atractores_validos]
    fig = go.Figure(data=go.Heatmap(
        z=matriz_probabilidades,
        x=etiquetas,
        y=etiquetas,
        colorscale='Magma',
        text=np.round(matriz_probabilidades, 1),
        texttemplate="%{text}%",
        textfont={"color": "white"}
    ))
    fig.update_layout(
        title='Máquina Guzmánica: Matriz de Transición de Estados (%)',
        xaxis_title='Estado de Destino (Hacia dónde salta)',
        yaxis_title='Estado de Origen (De dónde viene)',
        template='plotly_dark',
        width=700, height=700
    )
    print("Matriz de Markov generada.")
    fig.show()
matriz_transicion_guzmanica(1000)

Calculando cadena de Markov para 1000 saltos cuánticos...
Matriz de Markov generada.
